# Análisis Avanzado — Detección de Deslizamientos L4S
**Proyecto:** Landslide Detection ML — EAFIT  
**Autora:** Ana Patricia Montes Pimienta  

Este notebook implementa tres análisis avanzados sobre el dataset Landslide4Sense:
1. **SHAP** — Importancia de canales espectrales con valores Shapley
2. **Features Físicos** — TWI, Factor de Seguridad, Erodabilidad (análogo a PINNs)
3. **Incertidumbre RF** — Varianza epistémica entre árboles (análogo a MC Dropout)

---
> ⚠️ **Antes de correr:** cambia la variable `DRIVE_PATH` en la celda de configuración
> para que apunte a tu carpeta del proyecto en Google Drive.

## 0. Configuración inicial

In [ ]:
# ── Montar Google Drive ───────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print('✔ Drive montado')

In [ ]:
# ── ¡EDITA ESTA RUTA! ────────────────────────────────────────────────────
# Ruta a la carpeta raíz del proyecto dentro de tu Drive
# Ejemplo: '/content/drive/MyDrive/Landslide_ML'
DRIVE_PATH = '/content/drive/MyDrive/Landslide_ML'

import os, sys
from pathlib import Path

ROOT       = Path(DRIVE_PATH)
IMG_DIR    = ROOT / 'TrainData' / 'img'
MASK_DIR   = ROOT / 'TrainData' / 'mask'
SCRIPTS_DIR = ROOT / 'scripts'
OUT_DIR    = ROOT / 'results' / 'analisis_avanzado'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Añadir scripts al path para importar funciones
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SCRIPTS_DIR))

# Verificar que los datos existen
n_imgs  = len(list(IMG_DIR.glob('*.h5')))
n_masks = len(list(MASK_DIR.glob('*.h5')))
print(f'✔ Imágenes encontradas : {n_imgs}')
print(f'✔ Máscaras encontradas : {n_masks}')
assert n_imgs > 0, f'No se encontraron imágenes en {IMG_DIR}'
print(f'✔ Carpeta de salida    : {OUT_DIR}')

In [ ]:
# ── Instalar dependencias ─────────────────────────────────────────────────
# (shap no viene preinstalado en Colab)
%pip install shap --quiet
print('✔ shap instalado')

In [ ]:
# ── Imports globales ──────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.ndimage import uniform_filter
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import f1_score, roc_auc_score
from tqdm.notebook import tqdm   # barra de progreso visual en Colab
import shap

%matplotlib inline
plt.rcParams.update({
    'figure.dpi'     : 110,
    'font.family'    : 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('✔ Imports OK')

## 1. Carga de datos y extracción de features
Funciones compartidas por los tres análisis.

In [ ]:
# ── Constantes del dataset ────────────────────────────────────────────────
CHANNEL_NAMES = [
    'S2-B2 Azul',    'S2-B3 Verde',   'S2-B4 Rojo',
    'S2-B8 NIR',     'S2-B8A NIR-A',  'S2-B11 SWIR1',
    'S2-B12 SWIR2',  'S1-VV SAR',     'S1-VH SAR',
    'ALOS DEM',      'ALOS Pendiente','S2-B5 RedEdge1',
    'S2-B6 RedEdge2','S2-B7 RedEdge3',
]
CHANNEL_GROUPS = {
    'S2 Óptico': [0,1,2,3,4,5,6],
    'SAR':       [7,8],
    'DEM':       [9,10],
    'Red-Edge':  [11,12,13],
}
GROUP_COLORS = {
    'S2 Óptico': '#2E5FA3',
    'SAR':       '#D97706',
    'DEM':       '#16A34A',
    'Red-Edge':  '#9333EA',
}
CHANNEL_MEAN = [0.1245,0.1438,0.1312,0.2891,0.3015,0.2134,0.1789,
                0.0823,0.0641,0.4521,0.2189,0.3102,0.3478,0.3812]
CHANNEL_STD  = [0.0512,0.0621,0.0589,0.0934,0.0978,0.0734,0.0612,
                0.0341,0.0289,0.2134,0.1456,0.0812,0.0867,0.0923]

# Parámetros RF
N_SAMPLES    = 500    # ← aumenta si tienes RAM suficiente (max 3799)
N_ESTIMATORS = 200
SEED         = 42

In [ ]:
def normalize(patch):
    mean = np.array(CHANNEL_MEAN, dtype=np.float32).reshape(1,1,-1)
    std  = np.array(CHANNEL_STD,  dtype=np.float32).reshape(1,1,-1)
    return (patch - mean) / (std + 1e-8)

def extract_features(patch):
    """32 features por parche: 14 μ + 14 σ + NDVI + NDWI + SAR-CR + EVI."""
    eps = 1e-8
    nir, rojo, verde, azul = patch[:,:,3], patch[:,:,2], patch[:,:,1], patch[:,:,0]
    vv,  vh = patch[:,:,7], patch[:,:,8]
    ndvi  = (nir - rojo)  / (nir + rojo  + eps)
    ndwi  = (verde - nir) / (verde + nir + eps)
    sar_cr = vh / (vv + eps)
    evi    = 2.5*(nir-rojo) / (nir + 6*rojo - 7.5*azul + 1 + eps)
    return np.concatenate([
        patch.mean(axis=(0,1)), patch.std(axis=(0,1)),
        [ndvi.mean(), ndwi.mean(), sar_cr.mean(), evi.mean()]
    ]).astype(np.float32)

def build_feature_names():
    return ([f'μ_{n}' for n in CHANNEL_NAMES] +
            [f'σ_{n}' for n in CHANNEL_NAMES] +
            ['NDVI', 'NDWI', 'SAR-CR', 'EVI'])

def load_dataset(n_samples=N_SAMPLES, seed=SEED, return_patches=False):
    img_files = sorted(IMG_DIR.glob('*.h5'))
    rng = np.random.default_rng(seed)
    chosen = sorted(rng.choice(len(img_files), size=min(n_samples, len(img_files)), replace=False))
    X_list, y_list, patches_list, masks_list = [], [], [], []
    for i in tqdm(chosen, desc='Cargando parches'):
        img_f  = img_files[i]
        mask_f = MASK_DIR / img_f.name.replace('image_', 'mask_')
        if not mask_f.exists(): continue
        with h5py.File(img_f, 'r') as f:
            key = 'img' if 'img' in f else list(f.keys())[0]
            patch = f[key][()].astype(np.float32)
        with h5py.File(mask_f, 'r') as f:
            key = 'mask' if 'mask' in f else list(f.keys())[0]
            mask = f[key][()].astype(np.uint8)
        patch_n = normalize(patch)
        X_list.append(extract_features(patch_n))
        y_list.append(int(mask.max() > 0))
        if return_patches:
            patches_list.append(patch)
            masks_list.append(mask)
    X, y = np.stack(X_list), np.array(y_list)
    print(f'Dataset: {X.shape[0]} muestras | {y.sum()} positivos ({100*y.mean():.1f}%)')
    return (X, y, patches_list, masks_list) if return_patches else (X, y)

print('✔ Funciones de carga definidas')

In [ ]:
# Cargar dataset (con parches raw para mapa espacial)
X, y, patches, masks = load_dataset(N_SAMPLES, SEED, return_patches=True)
feature_names = build_feature_names()
print(f'Shape X: {X.shape}  |  Features: {len(feature_names)}')

In [ ]:
# Entrenar Random Forest base (usado en los tres análisis)
print('Entrenando Random Forest...')
rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=None,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED,
)
rf.fit(X, y)

cv_f1 = cross_val_score(rf, X, y, cv=5, scoring='f1', n_jobs=-1)
print(f'CV F1 (5-fold): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print('✔ Modelo entrenado')

---
## 2. Análisis SHAP — Importancia de canales espectrales
Los valores SHAP cuantifican la contribución marginal de cada canal a la predicción,
respetando las interacciones entre variables (a diferencia de la importancia por impureza del RF).

In [ ]:
# Calcular SHAP (TreeExplainer es exacto y eficiente para RF)
print('Calculando SHAP values...')
explainer   = shap.TreeExplainer(rf)
n_shap      = min(300, len(X))   # ← más muestras = más preciso pero más lento
X_shap      = X[:n_shap]
shap_values = explainer.shap_values(X_shap)

# shap_values es lista [clase_0, clase_1] → tomar clase positiva
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

print(f'SHAP shape: {sv.shape}  ({n_shap} muestras × {sv.shape[1]} features)')

In [ ]:
# ── Figura 2a: Bar chart importancia SHAP media (top-20) ─────────────────
mean_abs = np.abs(sv).mean(axis=0)
order    = np.argsort(mean_abs)[::-1][:20]

def feat_color(name):
    for g, idxs in CHANNEL_GROUPS.items():
        for i in idxs:
            if CHANNEL_NAMES[i].replace(' ','') in name.replace(' ',''):
                return GROUP_COLORS[g]
    if name in ('NDVI','EVI','NDWI'): return GROUP_COLORS['S2 Óptico']
    if name == 'SAR-CR': return GROUP_COLORS['SAR']
    return '#6B7280'

fig, ax = plt.subplots(figsize=(10, 7))
y_pos   = np.arange(len(order))
colors  = [feat_color(feature_names[i]) for i in order]

ax.barh(y_pos, mean_abs[order], color=colors, edgecolor='white', linewidth=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels([feature_names[i] for i in order], fontsize=10)
ax.invert_yaxis()
ax.set_xlabel('Importancia SHAP media |φᵢ|', fontsize=11)
ax.set_title('Importancia de features — Random Forest (SHAP) — Top 20',
             fontsize=13, fontweight='bold', pad=14)
ax.grid(axis='x', linestyle='--', alpha=0.4)

patches_leg = [mpatches.Patch(color=c, label=g) for g, c in GROUP_COLORS.items()]
ax.legend(handles=patches_leg, loc='lower right', fontsize=9, title='Sensor')

plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_importancia_features.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: shap_importancia_features.png')

In [ ]:
# ── Figura 2b: Importancia SHAP por grupo de sensor ───────────────────────
group_means, group_stds = {}, {}
for g, idxs in CHANNEL_GROUPS.items():
    g_feat_idx = [j for j, n in enumerate(feature_names)
                  if any(CHANNEL_NAMES[i].replace(' ','') in n.replace(' ','') for i in idxs)]
    if g_feat_idx:
        vals = mean_abs[g_feat_idx]
        group_means[g] = vals.mean()
        group_stds[g]  = vals.std()

# Añadir índices espectrales
idx_spec = [j for j, n in enumerate(feature_names) if n in ('NDVI','NDWI','SAR-CR','EVI')]
if idx_spec:
    group_means['Índices'] = mean_abs[idx_spec].mean()
    group_stds['Índices']  = mean_abs[idx_spec].std()

fig, ax = plt.subplots(figsize=(8, 5))
grupos  = list(group_means.keys())
vals    = [group_means[g] for g in grupos]
errs    = [group_stds[g]  for g in grupos]
cols    = [GROUP_COLORS.get(g, '#EF4444') for g in grupos]

bars = ax.bar(grupos, vals, yerr=errs, color=cols, edgecolor='white',
              linewidth=0.8, capsize=6, alpha=0.9)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0002,
            f'{v:.5f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('SHAP medio por canal del grupo', fontsize=11)
ax.set_title('Contribución SHAP por tipo de sensor\n'
             '(¿cuál sensor importa más para detectar deslizamientos?)',
             fontsize=11, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(OUT_DIR / 'shap_por_sensor.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: shap_por_sensor.png')

In [ ]:
# ── Figura 2c: Mapa de correlación entre canales (cross-talk) ─────────────
X_means = X[:, :14]   # solo las 14 medias por canal
corr    = np.corrcoef(X_means.T)
mask_tri = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(np.where(mask_tri, np.nan, corr), cmap='RdBu_r', vmin=-1, vmax=1)

short = [n.split(' ')[0] for n in CHANNEL_NAMES]
ax.set_xticks(range(14)); ax.set_yticks(range(14))
ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(short, fontsize=9)

for i in range(14):
    for j in range(14):
        if not mask_tri[i,j]:
            ax.text(j, i, f'{corr[i,j]:.2f}', ha='center', va='center',
                    fontsize=6.5, color='black' if abs(corr[i,j])<0.7 else 'white')

plt.colorbar(im, ax=ax, label='Correlación de Pearson', shrink=0.8)
ax.set_title('Correlación entre canales espectrales\n'
             '(análogo al análisis de cross-talk en FWI — Montoya et al., 2024)',
             fontsize=11, fontweight='bold', pad=12)
for b in [7, 9, 11]:  # límites entre grupos
    ax.axhline(b-0.5, color='white', lw=1.5)
    ax.axvline(b-0.5, color='white', lw=1.5)

plt.tight_layout()
plt.savefig(OUT_DIR / 'correlacion_canales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: correlacion_canales.png')

In [ ]:
# ── Resumen SHAP en texto ──────────────────────────────────────────────────
print('\n=== RESUMEN SHAP ===')
print(f'{"Rango":<6} {"Feature":<28} {"SHAP |φ| medio":>15}')
print('-'*52)
for rank, i in enumerate(order[:10], 1):
    print(f'{rank:<6} {feature_names[i]:<28} {mean_abs[i]:>15.6f}')
print()
print('Importancia por sensor:')
for g, m in sorted(group_means.items(), key=lambda x: -x[1]):
    bar = '█' * int(m * 8000)
    print(f'  {g:<18} {m:.5f}  {bar}')

---
## 3. Features Físicos — TWI y Factor de Seguridad
Derivados de los canales DEM y Pendiente. Inspirado en PINNs que incorporan
ecuaciones físicas como restricciones (Montoya et al., 2024).

In [ ]:
# ── Parámetros geotécnicos (suelos andinos tropicales) ────────────────────
PHI_DEG    = 28.0   # ángulo de fricción interna φ' (grados)
COHESION   = 12.0   # cohesión efectiva c' (kPa)
GAMMA      = 18.0   # peso unitario del suelo (kN/m³)
Z_PROF     = 2.0    # profundidad del plano de falla (m)
U_PRESION  = 5.0    # presión de poro (kPa) — condición húmeda

def compute_twi(dem, slope, kernel=9):
    """TWI = ln(A / tan(β))  — Beven & Kirkby (1979)"""
    area  = uniform_filter(dem.astype(np.float64), size=kernel) + 1e-6
    slope_rad = np.deg2rad(np.clip(slope * 60.0, 0.1, 89.9))
    return np.log(area / np.tan(slope_rad)).astype(np.float32)

def compute_fos(slope):
    """Factor de Seguridad infinito en pendiente — Skempton & DeLory (1957)"""
    slope_rad = np.deg2rad(np.clip(slope * 60.0, 0.1, 89.9))
    cos_b, sin_b = np.cos(slope_rad), np.sin(slope_rad)
    tan_phi = np.tan(np.deg2rad(PHI_DEG))
    fs_num  = COHESION + (GAMMA * Z_PROF * cos_b**2 - U_PRESION) * tan_phi
    fs_den  = GAMMA * Z_PROF * sin_b * cos_b + 1e-6
    return np.clip(fs_num / fs_den, 0.0, 5.0).astype(np.float32)

def physics_features_patch(patch):
    dem_ch   = patch[:, :, 9]
    slope_ch = patch[:, :, 10]
    vh_ch    = patch[:, :, 8]
    twi = compute_twi(dem_ch, slope_ch)
    fos = compute_fos(slope_ch)
    vh_norm  = (vh_ch  - vh_ch.min())  / (vh_ch.ptp()  + 1e-8)
    twi_norm = (twi    - twi.min())    / (twi.ptp()    + 1e-8)
    shi      = vh_norm / (twi_norm + 0.1)
    slope_deg = slope_ch * 60.0
    erod = slope_deg * np.std(vh_ch) * (vh_ch / (vh_ch.mean() + 1e-8))
    return {
        'twi': twi, 'fos': fos, 'shi': shi.astype(np.float32),
        'erod': erod.astype(np.float32),
        'twi_mean': float(twi.mean()),   'fos_min':    float(fos.min()),
        'fos_mean': float(fos.mean()),   'area_inest': float((fos<1.0).mean()),
        'shi_mean': float(shi.mean()),   'ero_mean':   float(erod.mean()),
        'twi_std':  float(twi.std()),    'fos_p25':    float(np.percentile(fos,25)),
    }

print('✔ Funciones de features físicos definidas')

In [ ]:
# ── Figura 3a: Mapa de features físicos para un parche ejemplo ────────────
# Buscar el primer parche positivo
ex_patch = ex_mask = None
for p, m in zip(patches, masks):
    if m.max() > 0:
        ex_patch, ex_mask = p, m
        break

phys_ex = physics_features_patch(ex_patch)
rgb = np.clip((ex_patch[:,:,[2,1,0]] - ex_patch[:,:,[2,1,0]].min()) /
              (ex_patch[:,:,[2,1,0]].ptp() + 1e-8), 0, 1)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
paneles = [
    (rgb,                       'RGB Falso Color (S2)',              'gray',     False),
    (ex_patch[:,:,9],           'ALOS DEM (normalizado)',            'terrain',  True),
    (ex_patch[:,:,10]*60,       'Pendiente (° aprox.)',              'YlOrRd',   True),
    (phys_ex['twi'],            'TWI — Índice Topográfico Humedad',  'Blues_r',  True),
    (phys_ex['fos'],            f'Factor de Seguridad FS\n'
                                f'(min={phys_ex["fos_min"]:.2f}  '
                                f'medio={phys_ex["fos_mean"]:.2f}  '
                                f'inestable={100*phys_ex["area_inest"]:.1f}%)',
                                'RdYlGn',  True),
    (ex_mask.astype(float),     'Máscara GT (deslizamiento)',        'Reds',     True),
]
for ax, (data, title, cmap, cbar) in zip(axes.flatten(), paneles):
    if title.startswith('RGB'):
        ax.imshow(rgb)
    else:
        im = ax.imshow(data, cmap=cmap)
        if cbar:
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=9.5, fontweight='bold')
    ax.axis('off')

fig.text(0.5, 0.01,
         f"φ'={PHI_DEG}°  c'={COHESION} kPa  γ={GAMMA} kN/m³  z={Z_PROF} m  u={U_PRESION} kPa",
         ha='center', fontsize=9, style='italic', color='#4B5563')
fig.suptitle('Features Físicos Geomecánicos — Parche Ejemplo L4S',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'mapa_features_fisicos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: mapa_features_fisicos.png')

In [ ]:
# ── Figura 3b: Distribución FS por clase + scatter TWI vs FS ─────────────
twi_vals, fos_vals, labels_phys = [], [], []
for patch, mask in tqdm(zip(patches, masks), total=len(patches), desc='Features físicos'):
    ph = physics_features_patch(patch)
    twi_vals.append(ph['twi_mean'])
    fos_vals.append(ph['fos_mean'])
    labels_phys.append(int(mask.max() > 0))

twi_arr  = np.array(twi_vals)
fos_arr  = np.array(fos_vals)
lab_arr  = np.array(labels_phys)
fos_pos  = fos_arr[lab_arr == 1]
fos_neg  = fos_arr[lab_arr == 0]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histograma FS
ax = axes[0]
bins = np.linspace(0, 3, 40)
ax.hist(fos_neg, bins=bins, alpha=0.7, color='#2E5FA3', label='Sin deslizamiento', density=True)
ax.hist(fos_pos, bins=bins, alpha=0.7, color='#D97706', label='Deslizamiento', density=True)
ax.axvline(1.0, color='red',    lw=1.5, ls='--', label='FS=1.0 (falla)')
ax.axvline(1.5, color='orange', lw=1.5, ls='--', label='FS=1.5 (crítico)')
ax.set_xlabel('Factor de Seguridad medio por parche', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title('Distribución del Factor de Seguridad por clase', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

# Scatter TWI vs FS
ax = axes[1]
ax.scatter(twi_arr[lab_arr==0], fos_arr[lab_arr==0], c='#2E5FA3', alpha=0.4, s=12,
           label='Sin deslizamiento', linewidths=0)
ax.scatter(twi_arr[lab_arr==1], fos_arr[lab_arr==1], c='#D97706', alpha=0.5, s=12,
           label='Deslizamiento', linewidths=0)
ax.axhline(1.0, color='red',    lw=1.2, ls='--', alpha=0.6)
ax.axhline(1.5, color='orange', lw=1.2, ls='--', alpha=0.6)
ax.set_xlabel('TWI medio por parche', fontsize=11)
ax.set_ylabel('Factor de Seguridad medio', fontsize=11)
ax.set_title('Espacio de Features Físicos: TWI vs. FS\n'
             '(separabilidad geomecánica del riesgo)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'twi_vs_fos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: twi_vs_fos.png')

In [ ]:
# ── Comparación RF: espectral vs. espectral+físico (5-fold CV) ────────────
print('Construyendo vectores de features físicos...')
X_phys_list = []
for patch, mask in tqdm(zip(patches, masks), total=len(patches), desc='Features'):
    ph = physics_features_patch(patch)
    X_phys_list.append(np.array([
        ph['twi_mean'], ph['twi_std'],
        ph['fos_min'],  ph['fos_mean'], ph['fos_p25'],
        ph['area_inest'], ph['shi_mean'], ph['ero_mean'],
    ]))

X_phys = np.stack(X_phys_list)
X_both = np.hstack([X, X_phys])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
results_phys = {}
for name, Xv in [('Espectral', X), ('Físico', X_phys), ('Espectral + Físico', X_both)]:
    rf_tmp = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                    min_samples_leaf=2, n_jobs=-1, random_state=SEED)
    f1s  = cross_val_score(rf_tmp, Xv, y, cv=skf, scoring='f1', n_jobs=-1)
    aucs = cross_val_score(rf_tmp, Xv, y, cv=skf, scoring='roc_auc', n_jobs=-1)
    results_phys[name] = {'f1': f1s.mean(), 'f1_s': f1s.std(),
                          'auc': aucs.mean(), 'auc_s': aucs.std()}
    print(f'  {name:<22}: F1={f1s.mean():.4f}±{f1s.std():.4f}  AUC={aucs.mean():.4f}±{aucs.std():.4f}')

# Figura de comparación
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
modelos = list(results_phys.keys())
x       = np.arange(len(modelos))
cols    = ['#2E5FA3', '#16A34A', '#D97706']

for ax, metric in zip(axes, ['f1', 'auc']):
    vals = [results_phys[m][metric]   for m in modelos]
    errs = [results_phys[m][metric+'_s'] for m in modelos]
    bars = ax.bar(x, vals, yerr=errs, color=cols, edgecolor='white',
                  linewidth=0.8, capsize=6, alpha=0.9)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_xticks(x); ax.set_xticklabels(modelos, fontsize=10)
    ax.set_ylim(0.5, 1.02)
    ax.set_ylabel('F1-Score' if metric=='f1' else 'AUC-ROC', fontsize=11)
    ax.set_title(f'Comparación {metric.upper()} — RF 5-fold CV', fontsize=11, fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.suptitle('Impacto de Features Físicos en el Rendimiento del RF',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'comparacion_features_fisicos.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: comparacion_features_fisicos.png')
mejora = results_phys['Espectral + Físico']['f1'] - results_phys['Espectral']['f1']
print(f'\n→ Mejora neta por features físicos: {mejora:+.4f} F1')

---
## 4. Incertidumbre Epistémica — Varianza entre árboles RF
Cada árbol del RF equivale a un modelo muestreado del posterior.
La varianza entre sus predicciones mide la incertidumbre del modelo
(análogo a Monte Carlo Dropout en redes neuronales).

In [ ]:
# ── Obtener predicciones de CADA árbol ────────────────────────────────────
print(f'Extrayendo predicciones de {rf.n_estimators} árboles...')
tree_preds = np.zeros((len(X), rf.n_estimators), dtype=np.float32)
for k, tree in enumerate(tqdm(rf.estimators_, desc='Árboles')):
    proba = tree.predict_proba(X)
    tree_preds[:, k] = proba[:,1] if proba.shape[1]==2 else float(tree.classes_[0])

# Métricas de incertidumbre
p_mean  = tree_preds.mean(axis=1)
p_var   = tree_preds.var(axis=1)
eps     = 1e-10
entropy = -(p_mean * np.log(p_mean+eps) + (1-p_mean) * np.log(1-p_mean+eps))
p_q05   = np.percentile(tree_preds, 5,  axis=1)
p_q95   = np.percentile(tree_preds, 95, axis=1)

y_pred = (p_mean >= 0.5).astype(int)
print(f'F1={f1_score(y, y_pred, zero_division=0):.4f}  '
      f'AUC={roc_auc_score(y, p_mean):.4f}')
print(f'Varianza media global: {p_var.mean():.6f}')
print(f'En deslizamientos    : {p_var[y==1].mean():.6f}')
print(f'En no-deslizamientos : {p_var[y==0].mean():.6f}')

In [ ]:
# ── Figura 4a: Distribución de incertidumbre por clase ────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pos_mask = y == 1; neg_mask = y == 0

for ax, (data, xlabel, title) in zip(axes, [
    (p_var,            'Varianza σ² entre árboles',    'Varianza epistémica'),
    (entropy,          'Entropía H(p)',                 'Entropía predictiva'),
    (p_q95 - p_q05,    'IC90% = P95 − P05',            'Amplitud IC 90%'),
]):
    bins = np.linspace(data.min(), data.max(), 40)
    ax.hist(data[neg_mask], bins=bins, alpha=0.7, color='#2E5FA3',
            label='Sin deslizamiento', density=True)
    ax.hist(data[pos_mask], bins=bins, alpha=0.7, color='#D97706',
            label='Deslizamiento', density=True)
    ax.axvline(data[pos_mask].mean(), color='#D97706', lw=1.5, ls='--',
               label=f'μ pos={data[pos_mask].mean():.4f}')
    ax.axvline(data[neg_mask].mean(), color='#2E5FA3', lw=1.5, ls='--',
               label=f'μ neg={data[neg_mask].mean():.4f}')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Densidad', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.legend(fontsize=7.5)

fig.suptitle('Distribución de Incertidumbre del RF por Clase',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'distribucion_incertidumbre.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: distribucion_incertidumbre.png')

In [ ]:
# ── Figura 4b: Curva de calibración ──────────────────────────────────────
n_bins = 10
bins   = np.linspace(0, 1, n_bins+1)
bin_c  = (bins[:-1] + bins[1:]) / 2
frac_obs, conf_mean, cnts = [], [], []
for lo, hi in zip(bins[:-1], bins[1:]):
    msk = (p_mean >= lo) & (p_mean < hi)
    n = msk.sum(); cnts.append(n)
    frac_obs.append(y[msk].mean() if n > 0 else np.nan)
    conf_mean.append(p_mean[msk].mean() if n > 0 else bin_c[len(conf_mean)])

frac_obs  = np.array(frac_obs)
conf_mean = np.array(conf_mean)
cnts      = np.array(cnts)
ece       = np.nansum(cnts/cnts.sum() * np.abs(frac_obs - conf_mean))

fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios':[2,1]})

ax = axes[0]
ax.plot([0,1],[0,1],'k--', lw=1, label='Calibración perfecta')
valid = ~np.isnan(frac_obs)
ax.plot(conf_mean[valid], frac_obs[valid], 'o-', color='#D97706',
        lw=2, ms=7, label=f'Random Forest (ECE={ece:.4f})')
ax.fill_between([0,1],[0,1],[1,1], alpha=0.07, color='red', label='Sobre-confianza')
ax.fill_between([0,1],[0,0],[0,1], alpha=0.07, color='blue', label='Infra-confianza')
ax.set_xlabel('Confianza predicha P(y=1)', fontsize=11)
ax.set_ylabel('Fracción positivos observados', fontsize=11)
ax.set_title(f'Reliability Diagram — ECE = {ece:.4f}',
             fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1)

ax = axes[1]
ax.hist(p_mean[y==0], bins=20, alpha=0.7, color='#2E5FA3',
        label='Sin deslizamiento', density=True)
ax.hist(p_mean[y==1], bins=20, alpha=0.7, color='#D97706',
        label='Deslizamiento', density=True)
ax.axvline(0.5, color='red', lw=1.2, ls='--', label='Umbral p=0.5')
ax.set_xlabel('Probabilidad predicha P(y=1)', fontsize=11)
ax.set_ylabel('Densidad', fontsize=11)
ax.set_title('Distribución de confianza por clase', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / 'curva_calibracion.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: curva_calibracion.png')

In [ ]:
# ── Figura 4c: Tasa de error vs. nivel de incertidumbre ──────────────────
errors     = (y_pred != y).astype(int)
var_bins   = np.linspace(0, p_var.max(), 11)
bin_centers = (var_bins[:-1] + var_bins[1:]) / 2
err_rates, cnts_v = [], []
for lo, hi in zip(var_bins[:-1], var_bins[1:]):
    msk = (p_var >= lo) & (p_var < hi)
    cnts_v.append(msk.sum())
    err_rates.append(errors[msk].mean() if msk.sum() > 0 else np.nan)

err_rates = np.array(err_rates)
corr_var_err = np.corrcoef(p_var, errors)[0,1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
colors_sc = ['#EF4444' if e else '#2E5FA3' for e in errors]
ax.scatter(p_var, p_mean, c=colors_sc, s=8, alpha=0.4, linewidths=0)
ax.axhline(0.5, color='gray', lw=1, ls='--')
ax.set_xlabel('Varianza epistémica σ²', fontsize=11)
ax.set_ylabel('Probabilidad media P(y=1)', fontsize=11)
ax.set_title('Varianza vs. Probabilidad\n(rojo = error de clasificación)',
             fontsize=11, fontweight='bold')
ax.legend(handles=[
    mpatches.Patch(color='#2E5FA3', label='Correcto'),
    mpatches.Patch(color='#EF4444', label='Error'),
], fontsize=9)

ax = axes[1]
valid = ~np.isnan(err_rates)
bar_cols = plt.cm.YlOrRd(np.linspace(0.2, 0.9, valid.sum()))
ax.bar(bin_centers[valid], err_rates[valid],
       width=(var_bins[1]-var_bins[0])*0.85,
       color=bar_cols, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Varianza epistémica σ² (bin)', fontsize=11)
ax.set_ylabel('Tasa de error', fontsize=11)
ax.set_title(f'Error vs. Incertidumbre  (r={corr_var_err:.3f})\n'
             '(alta varianza → mayor error = buen indicador)',
             fontsize=11, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(OUT_DIR / 'varianza_vs_error.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'✔ Guardado: varianza_vs_error.png  (r={corr_var_err:.3f})')

In [ ]:
# ── Figura 4d: Convergencia F1 y varianza vs. número de árboles ──────────
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
rf_full = RandomForestClassifier(n_estimators=N_ESTIMATORS, class_weight='balanced',
                                  min_samples_leaf=2, n_jobs=-1, random_state=SEED)
rf_full.fit(X_tr, y_tr)

checkpoints = list(range(10, N_ESTIMATORS+1, max(1, N_ESTIMATORS//25)))
f1_conv, var_conv = [], []
for k in tqdm(checkpoints, desc='Convergencia'):
    tp = np.zeros((len(X_te), k), dtype=np.float32)
    for i, tree in enumerate(rf_full.estimators_[:k]):
        pr = tree.predict_proba(X_te)
        tp[:,i] = pr[:,1] if pr.shape[1]==2 else float(tree.classes_[0])
    pm = tp.mean(axis=1)
    f1_conv.append(f1_score(y_te, (pm>=0.5).astype(int), zero_division=0))
    var_conv.append(tp.var(axis=1).mean())

fig, ax1 = plt.subplots(figsize=(10, 5))
ax2 = ax1.twinx()
ax1.plot(checkpoints, f1_conv, 'o-', color='#2E5FA3', ms=5, lw=2, label='F1-Score')
ax2.plot(checkpoints, var_conv, 's--', color='#D97706', ms=4, lw=1.8, alpha=0.9, label='Varianza σ²')
ax1.set_xlabel('Número de árboles K', fontsize=11)
ax1.set_ylabel('F1-Score', fontsize=11, color='#2E5FA3')
ax2.set_ylabel('Varianza epistémica media σ²', fontsize=11, color='#D97706')
ax1.tick_params(axis='y', labelcolor='#2E5FA3')
ax2.tick_params(axis='y', labelcolor='#D97706')
lines1, lbs1 = ax1.get_legend_handles_labels()
lines2, lbs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, lbs1+lbs2, fontsize=9, loc='lower right')
ax1.set_title('Convergencia RF: F1 y Varianza Epistémica vs. Número de Árboles',
              fontsize=12, fontweight='bold')
ax1.grid(axis='x', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'convergencia_arboles.png', dpi=150, bbox_inches='tight')
plt.show()
print('✔ Guardado: convergencia_arboles.png')

---
## 5. Resumen final

In [ ]:
# ── Resumen de todos los análisis ─────────────────────────────────────────
print('=' * 60)
print('  RESUMEN — ANÁLISIS AVANZADOS LANDSLIDE4SENSE')
print('=' * 60)

print('\n[1] SHAP — Top 5 features más importantes:')
for i in order[:5]:
    print(f'    {feature_names[i]:<28} {mean_abs[i]:.5f}')

print('\n    Sensor más importante:')
best_sensor = max(group_means, key=group_means.get)
print(f'    → {best_sensor}  (SHAP={group_means[best_sensor]:.5f})')

print('\n[2] Features físicos (RF 5-fold CV):')
for m, r in results_phys.items():
    print(f'    {m:<22}: F1={r["f1"]:.4f}  AUC={r["auc"]:.4f}')
print(f'    Mejora neta features físicos: {mejora:+.4f} F1')

print('\n[3] Incertidumbre epistémica:')
print(f'    ECE (calibración)          : {ece:.4f}')
print(f'    Corr. varianza-error       : r={corr_var_err:.3f}')
print(f'    Var. en deslizamientos     : {p_var[y==1].mean():.6f}')
print(f'    Var. en no-deslizamientos  : {p_var[y==0].mean():.6f}')

print(f'\n✅ Figuras guardadas en: {OUT_DIR}')
for f in sorted(OUT_DIR.glob('*.png')):
    print(f'   • {f.name}')